In [ ]:
import re, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import unicodedata

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from functools import lru_cache
from wordfreq import zipf_frequency

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

SEED = 42

# How the five models are combined:
# True  = rank-average (each model's scores become 0-1 positions, then averaged)
# False = probability average (each model's P(AI) averaged directly)
RANK_AVG = True
DATA_DIR = Path("data")
SUB_DIR = Path("submissions")
SUB_DIR.mkdir(exist_ok=True)


In [11]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

tr_df, val_df = train_test_split(train_df, test_size=2000,stratify=train_df["label"], random_state=SEED)

X_tr,  y_tr  = tr_df["text"].tolist(), tr_df["label"].to_numpy()
X_val, y_val = val_df["text"].tolist(), val_df["label"].to_numpy()
X_all, y_all = train_df["text"].tolist(), train_df["label"].to_numpy()

X_test = test_df["text"].tolist()

# 5-fold splitter, used for tuning settings inside the 8,000 training rows
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## Stylometric features
Each passage becomes 67 numbers describing *how* it is written, not *what* it is about:
sentence-length variation ("burstiness"), word length and readability, vocabulary richness,
how common the words are (wordfreq), punctuation rates, and function-word rates.

Whitespace is collapsed to single spaces first. In train.csv, blank lines appear in 66% of AI passages
and 0% of human ones. That is a formatting quirk of this dataset, not a writing style,
so we hide it from the model.


In [12]:
# A sentence ends at . ! or ? (plus any closing quote or bracket) followed by whitespace
SENT_SPLIT = re.compile(r"(?<=[.!?])[\"')\]]*\s+|\n+")
# A word is letters, optionally with an apostrophe part (don't, it's)
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

# Grammar words that carry little topic meaning
FUNCTION_WORDS = set("""
a about above after again against all am an and any are as at be because been before being below between both but by
can could did do does doing down during each few for from further had has have having he her here hers herself him
himself his how i if in into is it its itself just me more most my myself no nor not now of off on once only or other
our ours ourselves out over own same she should so some such than that the their theirs them themselves then there
these they this those through to too under until up very was we were what when where which while who whom why will
with would you your yours yourself yourselves may might must shall upon whether although though yet
""".split())
FIRST_PERSON = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves"}
SECOND_PERSON = {"you", "your", "yours", "yourself", "yourselves"}
HEDGES = {"may", "might", "could", "perhaps", "possibly", "likely", "potentially", "generally", "typically", "often"}
# Words that chat-style AI models tend to overuse
AI_CONNECTIVES = {"additionally", "furthermore", "moreover", "overall", "however", "ultimately", "notably",
                  "consequently", "therefore", "thus", "importantly", "significantly", "various", "crucial",
                  "essential", "ensure", "enhance", "comprehensive", "valuable", "unique"}

def collapse_text(t):
    # Turn any run of spaces, tabs or newlines into a single space
    return re.sub(r"\s+", " ", t).strip()

In [13]:
@lru_cache(maxsize=None)
def zipf(w):
    # How common a word is in English, from 0 (never seen) to about 7.7 ("the").
    # lru_cache remembers answers, so each word is looked up only once.
    return zipf_frequency(w, "en")

def syllables(w):
    # Rough syllable count: groups of vowels, minus a silent final "e"
    w = w.lower()
    n = len(re.findall(r"[aeiouy]+", w))
    if w.endswith("e") and n > 1 and not w.endswith(("le", "ee")):
        n -= 1
    return max(n, 1)

def safe_cv(x):
    # Coefficient of variation (std / mean); 0 when it can't be computed
    x = np.asarray(x, float)
    if len(x) > 1 and x.mean() > 0:
        return float(x.std() / x.mean())
    return 0.0

def mattr(tokens, w=50):
    # Moving-average type-token ratio: share of unique words in each 50-word window,
    # averaged over all windows (so long and short passages are comparable)
    if len(tokens) <= w:
        return len(set(tokens)) / max(len(tokens), 1)
    c = Counter(tokens[:w])
    vals = [len(c)]
    for i in range(w, len(tokens)):
        c[tokens[i]] += 1
        old = tokens[i - w]
        c[old] -= 1
        if c[old] == 0:
            del c[old]
        vals.append(len(c))
    return float(np.mean(vals)) / w

def lexical_burstiness(tokens, min_count=3):
    # Do repeated words come in clumps (> 0) or evenly spaced (< 0)?
    # Uses the gaps between repeats of each word that appears at least 3 times.
    positions = {}
    for i, t in enumerate(tokens):
        positions.setdefault(t, []).append(i)
    scores = []
    for p in positions.values():
        if len(p) >= min_count:
            gaps = np.diff(p).astype(float)
            m, s = gaps.mean(), gaps.std()
            if m + s > 0:
                scores.append((s - m) / (s + m))
    return float(np.mean(scores)) if scores else 0.0

In [14]:
def stylometric_features(text):
    f = {}
    n_chars = max(len(text), 1)
    sents = [s.strip() for s in SENT_SPLIT.split(text) if s and WORD_RE.search(s)]
    words = WORD_RE.findall(text)
    lw = [w.lower() for w in words]
    nw = max(len(words), 1)
    per100 = 100.0 / nw

    # Length
    f["n_words"] = len(words)

    # Sentence rhythm ("burstiness"): humans vary sentence length more
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], float)
    f["sent_len_mean"] = sl.mean()
    f["sent_len_std"] = sl.std()
    f["sent_len_cv"] = safe_cv(sl)
    f["sent_len_max"] = sl.max()
    f["sent_len_min"] = sl.min()
    f["frac_short_sents"] = float((sl < 8).mean())
    f["frac_long_sents"] = float((sl > 30).mean())
    f["sent_len_diff_mean"] = float(np.abs(np.diff(sl)).mean()) if len(sl) > 1 else 0.0

    # Word length and readability
    wl = np.array([len(w) for w in words] or [0], float)
    f["word_len_mean"] = wl.mean()
    f["word_len_std"] = wl.std()
    f["frac_long_words"] = float((wl >= 7).mean())
    syl = np.array([syllables(w) for w in words] or [1], float)
    f["syll_per_word"] = syl.mean()
    f["flesch"] = 206.835 - 1.015 * sl.mean() - 84.6 * syl.mean()
    f["fk_grade"] = 0.39 * sl.mean() + 11.8 * syl.mean() - 15.59

    # Vocabulary richness and repetition
    cnt = Counter(lw)
    f["ttr_first100"] = len(set(lw[:100])) / max(min(100, len(lw)), 1)
    f["mattr50"] = mattr(lw)
    f["hapax_ratio"] = sum(1 for c in cnt.values() if c == 1) / max(len(cnt), 1)
    s2 = sum(c * c for c in cnt.values())
    f["yule_k"] = 1e4 * (s2 - nw) / (nw * nw)
    f["lex_burstiness"] = lexical_burstiness(lw)
    bigrams = list(zip(lw, lw[1:]))
    bc = Counter(bigrams)
    f["repeated_bigram_frac"] = sum(c for c in bc.values() if c > 1) / max(len(bigrams), 1)
    first_words = [WORD_RE.findall(s)[0].lower() for s in sents]
    f["sent_start_diversity"] = len(set(first_words)) / max(len(first_words), 1)

    # Word predictability from wordfreq (Zipf scale: 0 = unseen word, about 7.7 = "the")
    z = np.array([zipf(w) for w in lw] or [0], float)
    f["zipf_mean"] = z.mean()
    f["zipf_std"] = z.std()
    f["frac_rare_words"] = float((z < 3).mean())
    f["frac_oov_words"] = float((z == 0).mean())
    content = np.array([zipf(w) for w in lw if w not in FUNCTION_WORDS] or [0], float)
    f["zipf_content_mean"] = content.mean()
    # How much predictability changes from sentence to sentence
    sent_z = [np.mean([zipf(w.lower()) for w in WORD_RE.findall(s)]) for s in sents]
    f["sent_zipf_std"] = float(np.std(sent_z)) if len(sent_z) > 1 else 0.0

    # Punctuation, per 100 words
    for name, ch in [("comma", ","), ("semicolon", ";"), ("colon", ":"), ("em_dash", "—"), ("en_dash", "–"),
                     ("paren", "("), ("exclam", "!"), ("question", "?"), ("period", ".")]:
        f[f"p_{name}"] = text.count(ch) * per100
    f["p_quote"] = (text.count('"') + text.count("“") + text.count("”")) * per100
    f["p_apostrophe"] = (text.count("'") + text.count("’")) * per100
    f["p_ellipsis"] = (text.count("...") + text.count("…")) * per100
    f["p_hyphen_word"] = len(re.findall(r"[A-Za-z]-[A-Za-z]", text)) * per100
    f["punct_per_sent"] = len(re.findall(r"[^\w\s]", text)) / max(len(sents), 1)

    # Character-level habits
    f["upper_frac"] = sum(c.isupper() for c in text) / n_chars
    f["digit_frac"] = sum(c.isdigit() for c in text) / n_chars
    f["non_ascii_frac"] = sum(ord(c) > 127 for c in text) / n_chars
    nfkc = unicodedata.normalize("NFKC", text)
    f["nfkc_changed_frac"] = sum(a != b for a, b in zip(text, nfkc)) / n_chars
    f["caps_word_frac"] = sum(1 for w in words if len(w) > 1 and w.isupper()) / nw
    f["titlecase_midsent_frac"] = sum(1 for a, b in zip(words, words[1:]) if b[0].isupper()) / nw

    # Function words and discourse markers, per 100 words
    f["function_word_rate"] = sum(w in FUNCTION_WORDS for w in lw) * per100
    f["first_person_rate"] = sum(w in FIRST_PERSON for w in lw) * per100
    f["second_person_rate"] = sum(w in SECOND_PERSON for w in lw) * per100
    f["contraction_rate"] = sum("'" in w for w in lw) * per100
    f["hedge_rate"] = sum(w in HEDGES for w in lw) * per100
    f["ai_connective_rate"] = sum(w in AI_CONNECTIVES for w in lw) * per100
    for w in ["the", "of", "and", "a", "to", "in", "is", "that", "it", "this", "with", "as", "but"]:
        f[f"fw_{w}"] = cnt.get(w, 0) * per100
    return f

In [15]:
def stylo_frame(texts):
    # One row per passage, one column per feature; whitespace collapsed first
    return pd.DataFrame([stylometric_features(collapse_text(t)) for t in texts])

t0 = time.time()
S_tr = stylo_frame(X_tr)
S_val = stylo_frame(X_val)
print(f"{S_tr.shape[1]} stylometric features | train {S_tr.shape} | val {S_val.shape} | {time.time() - t0:.0f}s")
S_tr.describe().T[["mean", "std", "min", "max"]].round(2).head(15)


67 stylometric features | train (8000, 67) | val (2000, 67) | 10s


,mean,std,min,max
n_words,204.79,60.01,2.00,390.00
sent_len_mean,26.56,29.25,1.00,339.00
sent_len_std,10.96,11.25,0.00,153.50
sent_len_cv,0.48,0.26,0.00,1.88
sent_len_max,48.32,36.43,1.00,339.00
sent_len_min,13.44,29.09,1.00,339.00
frac_short_sents,0.10,0.14,0.00,1.00
frac_long_sents,0.20,0.24,0.00,1.00
sent_len_diff_mean,12.94,17.76,0.00,307.00
word_len_mean,4.88,0.66,2.24,13.32


## Model 5: stylometric gradient boosting (HGB)
HistGradientBoosting builds a few hundred small decision trees. Each new tree corrects the mistakes
of the trees before it. Trees suit these features because the rules can be nonlinear and can combine
features (e.g. "short sentences AND few rare words → AI"), and the features don't need scaling.

We try 4 settings with 5-fold cross-validation on the 8,000 training rows, keep the best,
and score it once on validation.


In [16]:
# One row per model, for the comparison table at the end
results = []
# Each model's validation scores, kept for rank-averaging later
val_scores = {}

def record(name, cv_auc, val_score, params, seconds):
    val_scores[name] = val_score
    val_auc = roc_auc_score(y_val, val_score)
    results.append({"model": name, "cv_auc": round(cv_auc, 4), "val_auc": round(val_auc, 4),
                    "params": params, "seconds": round(seconds)})
    print(f"{name}: CV AUROC {cv_auc:.4f} | validation AUROC {val_auc:.4f} | {params} | {seconds:.0f}s")

# learning_rate: how much each new tree corrects the previous ones (smaller = more cautious)
# max_leaf_nodes: how many end points ("leaves") each tree can have (more = more complex rules)
HGB_GRID = {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [15, 31]}

t0 = time.time()
gs = GridSearchCV(HistGradientBoostingClassifier(max_iter=400, random_state=SEED),
                  HGB_GRID, scoring="roc_auc", cv=cv, n_jobs=-1)
gs.fit(S_tr, y_tr)

# GridSearchCV refits the best setting on all 8,000 training rows
stylo_hgb = gs.best_estimator_
stylo_hgb_params = gs.best_params_
# Column 1 of predict_proba is P(AI)
record("stylo_hgb", gs.best_score_, stylo_hgb.predict_proba(S_val)[:, 1], stylo_hgb_params, time.time() - t0)


stylo_hgb: CV AUROC 0.9606 | validation AUROC 0.9701 | {'learning_rate': 0.1, 'max_leaf_nodes': 31} | 14s


In [17]:
from sklearn.inspection import permutation_importance

# Shuffle one feature at a time on validation and measure how much AUROC drops.
# A big drop means the model depends on that feature.
perm = permutation_importance(stylo_hgb, S_val, y_val, scoring="roc_auc",
                              n_repeats=5, random_state=SEED, n_jobs=-1)

# "direction": does a higher value of this feature, on its own, point toward AI or human?
imp = pd.DataFrame({
    "importance": perm.importances_mean,
    "direction": ["AI higher" if roc_auc_score(y_val, S_val[c]) > 0.5 else "AI lower" for c in S_val.columns],
}, index=S_val.columns).sort_values("importance", ascending=False)
imp.round(4).head(15)


,importance,direction
repeated_bigram_frac,0.0343,AI lower
hapax_ratio,0.0245,AI higher
n_words,0.0234,AI lower
mattr50,0.0158,AI higher
p_colon,0.0114,AI higher
p_exclam,0.0062,AI higher
punct_per_sent,0.0058,AI higher
zipf_mean,0.0055,AI lower
fw_and,0.0052,AI lower
sent_len_std,0.0045,AI lower


## NB-SVM (Wang & Manning, 2012)
A logistic regression on n-gram features, with one extra step first. Each feature is multiplied
by its Naive Bayes log-count ratio:

    r = log( share of AI passages' n-grams that are this one / same share for human passages )

Features that mostly appear in AI text get a large positive r, human-typical features get a
large negative r, and features that show up equally in both get r ≈ 0. Logistic regression then
starts from features that are already scaled by how informative they are, which works especially
well on short texts.

r is computed inside `fit`, so during cross-validation it only ever sees that fold's training rows.


In [24]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression

# ClassifierMixin must come before BaseEstimator, or GridSearchCV treats it incorrectly
class NBLogReg(ClassifierMixin, BaseEstimator):
    def __init__(self, C=1.0, alpha=1.0):
        # C: regularization strength for the logistic regression (smaller = simpler model)
        # alpha: smoothing added to every count so no feature has a zero count
        self.C = C
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y)
        # Total count of each feature in AI passages (p) and in human passages (q)
        p = self.alpha + np.asarray(X[y == 1].sum(axis=0)).ravel()
        q = self.alpha + np.asarray(X[y == 0].sum(axis=0)).ravel()
        # Log ratio of each feature's share among AI vs among human features
        self.r_ = np.log((p / p.sum()) / (q / q.sum()))
        self.clf_ = LogisticRegression(C=self.C, max_iter=5000, solver="liblinear")
        self.clf_.fit(X.multiply(self.r_).tocsr(), y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, X):
        # The same r learned in fit is applied to new data
        return self.clf_.predict_proba(X.multiply(self.r_).tocsr())

    def predict(self, X):
        # Hard 0/1 labels; required by cross_val_predict even though we only use probabilities
        return self.clf_.predict(X.multiply(self.r_).tocsr())



In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0]

def run_specialist(name, make_vectorizer):
    t0 = time.time()
    # Learn the vocabulary from the 8,000 training rows only, then apply it to validation
    vec = make_vectorizer()
    A_tr = vec.fit_transform(X_tr)
    A_val = vec.transform(X_val)
    print(f"{name}: {A_tr.shape[1]:,} features")

    # Pick C by 5-fold CV on the training rows; the best C is refit on all 8,000
    gs = GridSearchCV(NBLogReg(), {"C": C_GRID}, scoring="roc_auc", cv=cv, n_jobs=-1)
    gs.fit(A_tr, y_tr)
    model = gs.best_estimator_
    record(name, gs.best_score_, model.predict_proba(A_val)[:, 1], gs.best_params_, time.time() - t0)
    return vec, model


## Models 1–4: four NB-SVM specialists
Each specialist reads a different part of the text, so they make different mistakes:

| Specialist | Reads | Example clue |
|---|---|---|
| Word | words and word pairs | "delve into" |
| Char | 3–5-letter pieces inside words | "-ization", typos |
| Skeleton | function words + punctuation only; other words → x / Xc / 0 | ", and it is x to x" |
| Cross-boundary char | 1–5-character pieces across spaces and punctuation | ", which", " -- " |

The skeleton and cross-boundary models were added because they depend less on topic words.


In [20]:
def word_vectorizer():
    # Words and word pairs that appear in at least 2 passages and in at most 95% of them
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                           binary=True, use_idf=False, norm=None)

word_vec, word_nb = run_specialist("nbsvm_word", word_vectorizer)


nbsvm_word: 167,947 features
nbsvm_word: CV AUROC 0.9436 | validation AUROC 0.9512 | {'C': 0.3} | 6s


In [21]:
def char_vectorizer():
    # char_wb = 3-5 character pieces taken inside word boundaries; keep the 300k most common
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=300_000,
                           binary=True, use_idf=False, norm=None)

char_vec, char_nb = run_specialist("nbsvm_char", char_vectorizer)


nbsvm_char: 132,980 features
nbsvm_char: CV AUROC 0.9440 | validation AUROC 0.9535 | {'C': 0.03} | 25s


In [22]:
# Tokens: words (with apostrophes), numbers, or single punctuation marks
TOK_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\sA-Za-z\d]")

def skeleton(text):
    out = []
    for tok in TOK_RE.findall(text):
        low = tok.lower()
        if low in FUNCTION_WORDS:
            # Keep grammar words as they are
            out.append(low)
        elif tok[0].isdigit():
            out.append("0")
        elif tok[0].isalpha():
            # Content word: keep only whether it was capitalized
            out.append("Xc" if tok[0].isupper() else "x")
        else:
            # Punctuation is kept
            out.append(tok)
    return " ".join(out)

print(skeleton("The stew offers a unique blend of flavors, and it is easy to make."))

def skeleton_vectorizer():
    # Read the skeleton as space-separated tokens, 1-4 at a time; keep case so x and Xc stay different
    return TfidfVectorizer(preprocessor=skeleton, token_pattern=r"\S+", lowercase=False,
                           ngram_range=(1, 4), min_df=3, binary=True, use_idf=False, norm=None)

skel_vec, skel_nb = run_specialist("nbsvm_skeleton", skeleton_vectorizer)

# Effective weight of each pattern = NB ratio × regression coefficient
eff = skel_nb.clf_.coef_.ravel() * skel_nb.r_
names = skel_vec.get_feature_names_out()
order = np.argsort(eff)
print("Toward AI:   ", " | ".join(names[order[-12:]][::-1]))
print("Toward human:", " | ".join(names[order[:12]]))


the x x a x x of x , and it is x to x .
nbsvm_skeleton: 90,953 features
nbsvm_skeleton: CV AUROC 0.9564 | validation AUROC 0.9668 | {'C': 0.1} | 9s
Toward AI:    … | / x / 0 | # # | from x . x | . > | in x to his | ! | : - | # Xc | not x what to | x x - 0 | 0 . in a
Toward human: ( ; | > < x / | Xc ( ; | the x 0 / | . < x / | that in x | a x x i | x Xc the x | , 0 ) , | $ - x . | ) 0 x Xc | < x / >


In [23]:
def charx_vectorizer():
    # analyzer="char" lets pieces run across spaces and punctuation; whitespace collapsed first
    # so blank lines can't be used; case kept
    return TfidfVectorizer(analyzer="char", preprocessor=collapse_text, lowercase=False, ngram_range=(1, 5),
                           min_df=3, max_features=300_000, binary=True, use_idf=False, norm=None)

charx_vec, charx_nb = run_specialist("nbsvm_charx", charx_vectorizer)


nbsvm_charx: 300,000 features
nbsvm_charx: CV AUROC 0.9663 | validation AUROC 0.9716 | {'C': 0.1} | 93s


## Combining the five models: rank-averaging
Each model's scores are turned into ranks between 0 and 1 (the lowest score gets about 0, the highest
gets 1), and the five rank columns are averaged. Ranks put all models on the same scale, and AUROC
only depends on the ordering anyway. The stylometric model gets an extra weight, because it is
the only model that reads style numbers rather than n-grams.

The weight is chosen using **out-of-fold scores on the 8,000 training rows**. Each training passage
gets scored by a model that did not train on it, so we can test weights without touching validation.

`RANK_AVG` in the first cell switches between this rank-average (`True`) and a plain average of the
models' probabilities (`False`). The style weight is chosen separately for whichever method is active.


In [26]:
from sklearn.model_selection import cross_val_predict
from scipy.stats import rankdata

t0 = time.time()
SPECIALISTS = {"nbsvm_word": (word_vec, word_nb), "nbsvm_char": (char_vec, char_nb),
               "nbsvm_skeleton": (skel_vec, skel_nb), "nbsvm_charx": (charx_vec, charx_nb)}

# For each model: train on 4 folds, score the 5th, repeat 5 times.
# Every training passage ends up with a score from a model that never saw it.
oof_scores = {}
for name, (vec, model) in SPECIALISTS.items():
    A_tr = vec.transform(X_tr)
    oof_scores[name] = cross_val_predict(NBLogReg(C=model.C), A_tr, y_tr, cv=cv,
                                         method="predict_proba", n_jobs=-1)[:, 1]
oof_scores["stylo_hgb"] = cross_val_predict(
    HistGradientBoostingClassifier(max_iter=400, random_state=SEED, **stylo_hgb_params),
    S_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

for name, s in oof_scores.items():
    print(f"{name:15s} out-of-fold AUROC {roc_auc_score(y_tr, s):.4f}")

def rank01(x):
    # Scores -> ranks scaled to (0, 1]
    return rankdata(x) / len(x)

def rank_avg(scores, names, style_w):
    # Weighted average of each model's rank; the stylometric model counts style_w times
    weights = [style_w if n == "stylo_hgb" else 1.0 for n in names]
    return np.average([rank01(scores[n]) for n in names], axis=0, weights=weights)

def prob_avg(scores, names, style_w):
    # Weighted average of each model's P(AI); the stylometric model counts style_w times
    weights = [style_w if n == "stylo_hgb" else 1.0 for n in names]
    return np.average([scores[n] for n in names], axis=0, weights=weights)

def combine(scores, names, style_w):
    # Use whichever method RANK_AVG (set in the first cell) selects
    if RANK_AVG:
        return rank_avg(scores, names, style_w)
    return prob_avg(scores, names, style_w)

COMBINE_NAME = "rank_average" if RANK_AVG else "prob_average"
MODELS = ["nbsvm_word", "nbsvm_char", "nbsvm_skeleton", "nbsvm_charx", "stylo_hgb"]
STYLE_W_GRID = [1, 2, 3, 4, 5]

# Try each weight on the out-of-fold scores and keep the best
oof_auc = {w: roc_auc_score(y_tr, combine(oof_scores, MODELS, w)) for w in STYLE_W_GRID}
for w, a in oof_auc.items():
    print(f"style weight {w}: out-of-fold AUROC {a:.4f}")
STYLE_W = max(oof_auc, key=oof_auc.get)

# Only now apply the chosen weight to validation
record(COMBINE_NAME, oof_auc[STYLE_W], combine(val_scores, MODELS, STYLE_W),
       {"style_w": STYLE_W}, time.time() - t0)


nbsvm_word      out-of-fold AUROC 0.9435
nbsvm_char      out-of-fold AUROC 0.9441
nbsvm_skeleton  out-of-fold AUROC 0.9564
nbsvm_charx     out-of-fold AUROC 0.9664
stylo_hgb       out-of-fold AUROC 0.9605
style weight 1: out-of-fold AUROC 0.9797
style weight 2: out-of-fold AUROC 0.9827
style weight 3: out-of-fold AUROC 0.9834
style weight 4: out-of-fold AUROC 0.9831
style weight 5: out-of-fold AUROC 0.9825
rank_average: CV AUROC 0.9834 | validation AUROC 0.9899 | {'style_w': 3} | 103s


In [27]:
display(pd.DataFrame(results).drop(columns="seconds"))

# How much does validation AUROC drop when each model is removed from the average?
full_auc = roc_auc_score(y_val, combine(val_scores, MODELS, STYLE_W))
rows = []
for m in MODELS:
    rest = [n for n in MODELS if n != m]
    rows.append({"left out": m, "val_auc": roc_auc_score(y_val, combine(val_scores, rest, STYLE_W))})
ablation = pd.DataFrame(rows)
ablation["change"] = ablation["val_auc"] - full_auc
ablation.round(4)


,model,cv_auc,val_auc,params
0,stylo_hgb,0.9606,0.9701,"{'learning_rate': 0.1, 'max_leaf_nodes': 31}"
1,nbsvm_word,0.9436,0.9512,{'C': 0.3}
2,nbsvm_char,0.9440,0.9535,{'C': 0.03}
3,nbsvm_skeleton,0.9564,0.9668,{'C': 0.1}
4,nbsvm_charx,0.9663,0.9716,{'C': 0.1}
5,rank_average,0.9834,0.9899,{'style_w': 3}


,left out,val_auc,change
0,nbsvm_word,0.9892,-0.0008
1,nbsvm_char,0.9899,-0.0000
2,nbsvm_skeleton,0.9885,-0.0014
3,nbsvm_charx,0.9897,-0.0003
4,stylo_hgb,0.9787,-0.0113


## Final model: refit on all 10,000 labeled rows and predict test.csv
The validation set has done its job. For the final predictions, every model is refit on all
10,000 labeled rows with the settings chosen above (each specialist's C, the HGB settings and the
style weight). Nothing is re-tuned here.

test.csv is only ever *transformed* and *predicted*. No vocabulary, NB ratio or model is fit on it.


In [28]:
t0 = time.time()
VECTORIZERS = {"nbsvm_word": word_vectorizer, "nbsvm_char": char_vectorizer,
               "nbsvm_skeleton": skeleton_vectorizer, "nbsvm_charx": charx_vectorizer}

test_scores = {}
for name, make_vectorizer in VECTORIZERS.items():
    # New vectorizer: vocabulary learned from all 10,000 labeled rows, then applied to test
    vec = make_vectorizer()
    A_all = vec.fit_transform(X_all)
    A_test = vec.transform(X_test)
    # Reuse the C chosen by cross-validation earlier
    best_C = SPECIALISTS[name][1].C
    model = NBLogReg(C=best_C).fit(A_all, y_all)
    test_scores[name] = model.predict_proba(A_test)[:, 1]
    print(f"{name:15s} refit on {len(X_all):,} rows (C={best_C}) | {time.time() - t0:.0f}s")

# Stylometric model: features for all labeled rows and for test, same settings as before
S_all = stylo_frame(X_all)
S_test = stylo_frame(X_test)
stylo_final = HistGradientBoostingClassifier(max_iter=400, random_state=SEED, **stylo_hgb_params)
stylo_final.fit(S_all, y_all)
test_scores["stylo_hgb"] = stylo_final.predict_proba(S_test)[:, 1]
print(f"stylo_hgb       refit on {len(X_all):,} rows | {time.time() - t0:.0f}s")

# Same combining method and style weight as on validation
test_pred = combine(test_scores, MODELS, STYLE_W)


nbsvm_word      refit on 10,000 rows (C=0.3) | 3s
nbsvm_char      refit on 10,000 rows (C=0.03) | 17s
nbsvm_skeleton  refit on 10,000 rows (C=0.1) | 23s
nbsvm_charx     refit on 10,000 rows (C=0.1) | 54s
stylo_hgb       refit on 10,000 rows | 71s


In [29]:
sub = pd.DataFrame({"id": test_df["id"], "label": test_pred})
# New file name, so earlier submissions are not overwritten
out_path = SUB_DIR / f"{COMBINE_NAME}_final.csv"
sub.to_csv(out_path, index=False)

# Check the file against Kaggle's sample: same columns, same ids in the same order, scores in [0, 1]
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) and (sub["id"].values == sample["id"].values).all()
assert sub["label"].between(0, 1).all() and sub["label"].notna().all()

print(f"wrote {out_path}: {len(sub):,} rows | mean score {sub['label'].mean():.3f}")
sub.head()


wrote submissions/rank_average_final.csv: 3,000 rows | mean score 0.500


,id,label
0,13001,0.901048
1,13002,0.295714
2,13003,0.453048
3,13004,0.425571
4,13005,0.942095


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
comparison = pd.DataFrame(results)[["model", "cv_auc"]].copy()
comparison["val_auc"] = [roc_auc_score(y_val, val_scores[name]) for name in comparison.model]
comparison.to_csv(results_dir / "model_comparison.csv", index=False)
ablation.to_csv(results_dir / "ablation.csv", index=False)
pd.DataFrame({"style_weight": list(oof_auc), "oof_auc": list(oof_auc.values())}).to_csv(
    results_dir / "style_weight_search.csv", index=False)
validation = pd.DataFrame({"id": val_df.id.to_numpy(), "label": y_val, **val_scores})
validation.to_csv(results_dir / "validation_scores.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 5))
for name in [*MODELS, COMBINE_NAME]:
    fpr, tpr, _ = roc_curve(y_val, val_scores[name])
    ax.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_val, val_scores[name]):.4f})")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Validation ROC on 2,000 passages")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(results_dir / "validation_roc.png", dpi=160)
plt.show()